In [1]:

import os
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. UCI-HAR Download
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"

if os.path.exists(DATA_PATH):

    print("Existing dataset found")
    print(DATA_PATH)

else:

    print("Downloading UCI-HAR dataset...")

    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00240/UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content/drive/MyDrive/Colab Notebooks")

    print("Download completed")


# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 4. Load dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # label 1~6 -> 0~5
    y = y - 1

    return X, y


# Official train/test split
X_train, y_train = load_data("train")
X_test, y_test = load_data("test")


print("\nDataset shape")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


# ============================================================
# 5. Normalization
# ============================================================

mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = (X_train - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)


# ============================================================
# 6. One-hot encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)


# ============================================================
# 7. CNN Model
# ============================================================

inputs = Input(shape=(128, 9))


conv1 = Conv1D(
    filters=32,
    kernel_size=3,
    activation="relu"
)(inputs)

pool1 = MaxPooling1D(
    pool_size=2
)(conv1)


conv2 = Conv1D(
    filters=64,
    kernel_size=3,
    activation="relu"
)(pool1)

pool2 = MaxPooling1D(
    pool_size=2
)(conv2)


flat = Flatten()(pool2)


dense = Dense(
    128,
    activation="relu"
)(flat)


outputs = Dense(
    n_outputs,
    activation="softmax"
)(dense)


# ============================================================
# 8. Model
# ============================================================

model = Model(
    inputs=inputs,
    outputs=outputs
)


model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


model.summary()


# ============================================================
# 9. Train
# ============================================================

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    shuffle=True,
    verbose=1
)


# ============================================================
# 10. Prediction
# ============================================================

y_prob = model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 11. Evaluation
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n==============================")
print("TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")


# ============================================================
# 12. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 13. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)


print("\n==============================")
print("CONFUSION MATRIX")
print("==============================")

print(cm)


Existing dataset found
/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset

Dataset shape
X_train: (7352, 128, 9)
y_train: (7352,)
X_test : (2947, 128, 9)
y_test : (2947,)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 9)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 126, 32)        │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 63, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 61, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       245,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 253,766 (991.27 KB)

 Trainable params: 253,766 (991.27 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 45ms/step - accuracy: 0.8327 - loss: 0.4527 - val_accuracy: 0.9177 - val_loss: 0.4070
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9532 - loss: 0.1111 - val_accuracy: 0.9157 - val_loss: 0.4377
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9594 - loss: 0.0889 - val_accuracy: 0.9143 - val_loss: 0.4589
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9623 - loss: 0.0797 - val_accuracy: 0.9123 - val_loss: 0.4673
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9641 - loss: 0.0741 - val_accuracy: 0.9103 - val_loss: 0.4902
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9660 - loss: 0.0693 - val_accuracy: 0.9082 - val_loss: 0.5002
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9687 - loss: 0.0645 - val_accuracy: 0.9055 - val_loss: 0.5276
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9713 - loss: 0.0592 - val_accuracy: 0.9028 - val_loss